# Protein Foundation Model Benchmark

This notebook demonstrates the usage of the Protein Foundation Model Benchmark Framework.

## Overview

1. Setup environment and configuration
2. Load pretrained protein foundation models
3. Load and preprocess benchmark datasets
4. Run benchmarking pipeline
5. Evaluate and visualize results
6. Export results for publication

## Requirements

Run the setup cell to install dependencies.

In [ ]:
# Install framework if running outside the repo
# !pip install -e ..

# Imports
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, "..")

from src import BenchmarkFramework
from src.utils.environment import log_environment_info
from src.utils.seed import set_seed
from src.utils.logging import get_logger, setup_logging

In [ ]:
# Initialize the framework
framework = BenchmarkFramework(
    project_root="..",
    seed=42,
    log_level="INFO",
)

# Check environment
env_info = log_environment_info()
print(f"Environment: {env_info['platform']['system']}")
print(f"PyTorch: {env_info['pytorch_version']}")
print(f"GPUs: {env_info['num_gpus']}")

# Check dependencies
deps = framework.check_dependencies()
missing_deps = [k for k, v in deps.items() if not v]
if missing_deps:
    print(f"Warning: Missing dependencies: {missing_deps}")

In [ ]:
# Create an experiment
experiment = framework.create_experiment(
    name="esm2_benchmark_fluorescence",
    description="Benchmark ESM-2 variants on fluorescence prediction",
    models=["esm2_t6_8M_UR50D", "esm2_t12_35M_UR50D", "esm2_t30_150M_UR50D"],
    datasets=["fluorescence"],
    task_types=["regression"],
    tags=["esm2", "fluorescence", "size-scaling"],
    device="auto",
)
print(f"Created experiment: {experiment.name} (ID: {experiment.experiment_id})")

In [ ]:
# Define custom pipeline runner
def run_pipeline(pipeline_manager):
    """Custom pipeline runner."""
    from src.models import ESM2
    from src.managers.pipeline_manager import PipelineConfig
    
    # Load model
    model = ESM2.from_pretrained(
        f"facebook/{pipeline_manager.config.model_name}",
        device=pipeline_manager.config.device,
    )
    pipeline_manager.load_model(model)
    
    # Load datasets
    dataset_manager = pipeline_manager.dataset_manager
    
    # Register dataset if needed
    # dataset_manager.register_dataset(...)
    
    # Get dataloaders (datasets must exist on disk)
    # train_loader, val_loader, test_loader = dataset_manager.get_train_val_test_loaders(
    #     pipeline_manager.config.dataset_name,
    #     tokenizer=model.get_tokenizer(),
    # )
    
    # For demo purposes, create dummy loaders
    # TODO: Replace with actual dataset loading
    print(f"Pipeline configured for: {pipeline_manager.config.model_name} on {pipeline_manager.config.dataset_name}")
    
    # Return pipeline config as placeholder
    from dataclasses import asdict
    return asdict(pipeline_manager.config)

In [ ]:
# Run benchmark
model_dataset_pairs = [
    ("esm2_t6_8M_UR50D", "fluorescence", "regression"),
    ("esm2_t12_35M_UR50D", "fluorescence", "regression"),
    ("esm2_t30_150M_UR50D", "fluorescence", "regression"),
]

results = framework.run_benchmark(
    experiment_id=experiment.experiment_id,
    model_dataset_pairs=model_dataset_pairs,
    pipeline_runner=run_pipeline,
)
print(f"Benchmark complete: {len(results)} runs")

In [ ]:
# Export results
export_path = framework.export_results(
    experiment_id=experiment.experiment_id,
    formats=["csv", "json"],
)
print(f"Exported to: {export_path}")

In [ ]:
# List all experiments
experiments = framework.list_experiments()
print(f"Total experiments: {len(experiments)}")
for exp in experiments:
    print(f"  - {exp.name} ({exp.status.value}): {exp.experiment_id}")

## Next Steps

1. **Add actual datasets**: Place datasets in `outputs/datasets/` and register them
2. **Download model checkpoints**: The framework will cache them automatically
3. **Run full pipeline**: Uncomment the dataset loading and training code
4. **Visualize results**: Use `src.plotting.plots` for publication figures